# Re-weight OMR Marks (Final A) - ECON2102 (2026, S1)

by [T. Kam](https://phantomachine.github.io/)

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

Purpose:

1. Import OMR scanner report - (``input.xlsx`` file)

2. Match/compare and convert ``string`` data in table of OMR Report (responses with green/pink or CORRECT/WRONG table)

3. Export to new ``output.xlsx`` file

## Specify file path(s) and names

In [ ]:
# Users can modify this!

# path
filepath = "data/final/"

# Source file
source_name = "ECON2102_MCQ report_S1_2026 Final"
file_source = filepath + source_name + ".xlsx"

# Target output file
file_out = filepath + source_name + "-reweighted" + ".xlsx"
stats_out = filepath + source_name + "-reweighted-stats" + ".xlsx"

# Specify your question weights / scores
weights = [ 5, 5, 5, 5, 5, 2, 10, 2, 6, 5 ]
 
# What stats would you like to report?
stat_au_choix = [ np.mean, np.median, np.std, min, max ]

## Import OMR report (xlsx format)

In [ ]:
# See file_source for where the table row begins!
start_row = 5

# Import as dataframe
df = pd.read_excel( file_source, 
                    sheet_name="Student Response Report", 
                    header=start_row )

In [ ]:
# Quick peek 
df

## Clone, clean, set index to ``UNum``

In [ ]:
# Clone to new dataframe dg, drop last row (Mean)
dg = df.copy()
dg.drop(dg.index[-1], inplace=True)

In [ ]:
# Inspect key names
dg.keys()

In [ ]:
dg = dg.drop(["Total", "Percent", "Grade"], axis=1)

In [ ]:
# Set series "UNum" as index
dg = dg.set_index('UNum')
dg

## Get ``Answer Key`` and create comparison dataframe ``dt``

In [ ]:
dg.loc['Answer Key',:].values

In [ ]:
# make array tiled from Answer Key row
d_testval = np.tile(dg.loc['Answer Key', :].values, (dg.shape[0], 1))
# create test dataframe with values from d_testval
dt = dg.copy()
dt.loc[:, :] = d_testval

In [ ]:
dt

## Equality test (boolean) ``df_bool``

In [ ]:
# Test equality between element-wise values of dg and dt
df_bool = dg.eq(dt)
df_bool

## Compute re-weighted index based on ``weights`` (user specified)

In [ ]:
# Custom weights matrix
W = np.tile(weights, (df_bool.shape[0], 1))

# Generate re-weighted OMR results
df_reweighted = df_bool*W
df_reweighted

## Write output to Excel

In [ ]:
# Some lipstick on this piggy
def pink_or_green(val):
    if val == 0:
        color = 'pink' 
    else:
        color = 'turquoise'
    return 'background-color: %s' % color

Add totals and percentage totals columns:

In [ ]:
# Maximal mark that can be earned
max_total = df_reweighted.loc["Answer Key"].sum()

# series: totals
totals = df_reweighted.sum(axis=1)

# series: percentage totals
totals_percent = 100*totals/max_total

In [ ]:
# Add new columns
df_reweighted["Total"] = totals
df_reweighted["Percent"] = totals_percent.astype(int)

In [ ]:
df_reweighted.style.apply(lambda x: x.map(pink_or_green), axis=None).to_excel(
    file_out, sheet_name="Scores")

In [ ]:
# Write to excel
# df_reweighted.style.applymap(pink_or_green).to_excel(file_out, sheet_name="Scores")

Below is what you should see in the output file.

In [ ]:
df_reweighted.style.apply(lambda x: x.map(pink_or_green), axis=None)

## Summary statistics

In [ ]:

# Drop answer key (so you don't bias up the stats!)
df_reweighted_nomaster = df_reweighted.drop(index=("Answer Key"))

# Compute basic stats
df_stats = df_reweighted_nomaster.apply(stat_au_choix).round(1)
df_stats

In [ ]:
# Write to new excel file
df_stats.to_excel(stats_out, sheet_name="Statistics")

## Distribution by question and total

In [ ]:
df_reweighted_nomaster.hist(bins=20);

In [ ]:
# k = len(df_reweighted_nomaster.columns)
# n = 3
# m = (k - 1) // n + 1
# fig, axes = plt.subplots(m, n, figsize=(n * 3, m * 2))
# for i, (key, series) in enumerate(df_reweighted_nomaster.iteritems()):
#     r, c = i // n, i % n
#     ax = axes[r, c]
#     series.hist(ax=ax)
#     ax2 = series.plot.kde(ax=ax, secondary_y=True, title=key)
#     ax2.set_ylim(0)

# fig.tight_layout()

## Kolmogorov-Smirnov test

Use default of null hypothesis that the empirical distribution is identical to a Normal distribution. The alternative is that they are not identical.

In [ ]:
# Run K-S test against a normal distro
result = stats.kstest(df_reweighted_nomaster["Total"], 
                                                    stats.norm.cdf)

# Make a dict of tuple, results, parse as pandas df
val = dict(zip(("Test Statistic", "p-value"), result))
df_ks = pd.DataFrame.from_dict(val, orient="index", columns=["value",])

# Message
if df_ks.loc["p-value"].item() <= 0.05:
    print("Reject null of normality!")
else:
    print("Cannot reject null (at 95% confidence): You're normal like all of us!")
    
# Show table
df_ks